# Semantic description ↔ 20D video point-cloud alignment

This notebook is a Colab-first research workflow for testing whether channel-level language descriptions preserve the same geometry as channel-level video-title embedding point clouds.

The workflow intentionally starts from the **20D reduced video embedding artifact** before any language-model work. It then uses the Google Colab native Gemini integration (`google.colab.ai`) to describe each channel, stores cached per-channel and overall results in Google Drive, and evaluates multiple distance/shape/ordering hypotheses.

## Research questions

1. **Relative distance alignment:** if channel A's 20D video point cloud is closest to channel B's point cloud, is channel A's generated description also closest to channel B's generated description?
2. **Shape alignment:** do broad, overlapping, or concentrated point clouds produce descriptions whose semantic segments are broad, overlapping, or concentrated?
3. **Narrative centrality:** do central concepts appear earlier and more prominently in the segmented description, expanding outward as the description proceeds?

## 0. Environment setup

This notebook expects to run in Google Colab. It mounts Google Drive, installs only lightweight analysis dependencies, and keeps paths centralized so every generated artifact is reproducible and reusable.

The native Gemini integration is imported from `google.colab.ai`, so no API key or Colab secret is required. If this import fails, the notebook will stop before generating descriptions rather than silently switching to a keyed API.

In [ ]:
# Install analysis dependencies. Colab usually already has most of these, but pinning nothing keeps the
# notebook compatible with the managed runtime.
%pip -q install sentence-transformers umap-learn seaborn scipy scikit-learn pandas numpy matplotlib tqdm

Mount Drive and define the canonical input/output paths. The main input is the 20D video embedding CSV produced upstream by Graphiko. The optimized clustered bundle is supported as a fallback because it also carries `embedding_20d` for each video.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cdist
from scipy.stats import kendalltau, pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import cosine_distances
from tqdm.auto import tqdm

try:
    from google.colab import drive
    from google.colab import ai
except Exception as exc:  # noqa: BLE001 - fail loudly with a Colab-specific explanation.
    raise RuntimeError(
        "This notebook must run in Google Colab because it uses the native "
        "google.colab.ai Gemini integration and Google Drive mount."
    ) from exc

from sentence_transformers import SentenceTransformer

sns.set_theme(style="whitegrid", context="notebook")

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
GRAPHIKO_ROOT = DRIVE_ROOT / "Graphiko"

PRIMARY_20D_CSV = GRAPHIKO_ROOT / "exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv"
FALLBACK_CLUSTER_BUNDLE = GRAPHIKO_ROOT / "exports/video_embeddings_clustered/optimized/business_cluster_video_embeddings_channel_optimized_bundle.json"

RESEARCH_ROOT = DRIVE_ROOT / "Graphiko/research/semantic_description_pointcloud_alignment"
CHANNEL_DESCRIPTION_DIR = RESEARCH_ROOT / "descriptions/by_channel"
OVERALL_DESCRIPTION_PATH = RESEARCH_ROOT / "descriptions/overall/overall_description.json"
SEGMENT_DIR = RESEARCH_ROOT / "segments/by_channel"
RESULTS_DIR = RESEARCH_ROOT / "results/latest"
PLOTS_DIR = RESULTS_DIR / "plots"
TABLES_DIR = RESULTS_DIR / "tables"

for path in [CHANNEL_DESCRIPTION_DIR, OVERALL_DESCRIPTION_PATH.parent, SEGMENT_DIR, PLOTS_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()
print("Research outputs:", RESEARCH_ROOT)

## 1. Load the 20D video embedding artifact first

This section enforces the requested ordering: no LLM calls happen until the 20D video embeddings are loaded and validated. The loader supports both the canonical reduced CSV and the optimized clustering bundle, normalizing either source into the same table with one row per video and exactly 20 numeric embedding columns.

In [ ]:
def _embedding_columns(df: pd.DataFrame) -> list[str]:
    reduced = [f"embedding_reduced_{i:02d}" for i in range(1, 21)]
    if all(c in df.columns for c in reduced):
        return reduced
    generic = [f"embedding_{i:02d}" for i in range(1, 21)]
    if all(c in df.columns for c in generic):
        return generic
    dim = [f"dim_{i}" for i in range(20)]
    if all(c in df.columns for c in dim):
        return dim
    raise ValueError("Could not find a stable set of 20 embedding columns.")


def load_20d_video_embeddings() -> tuple[pd.DataFrame, list[str], dict[str, Any]]:
    """Load the canonical 20D video embedding artifact, with a JSON bundle fallback."""
    if PRIMARY_20D_CSV.exists():
        df = pd.read_csv(PRIMARY_20D_CSV)
        emb_cols = _embedding_columns(df)
        source = {"kind": "reduced_csv", "path": str(PRIMARY_20D_CSV)}
    elif FALLBACK_CLUSTER_BUNDLE.exists():
        with FALLBACK_CLUSTER_BUNDLE.open("r", encoding="utf-8") as fh:
            bundle = json.load(fh)
        records = bundle.get("artifacts", {}).get("videos_clustered", [])
        df = pd.DataFrame(records)
        if "embedding_20d" not in df.columns:
            raise ValueError(f"Fallback bundle exists but has no embedding_20d field: {FALLBACK_CLUSTER_BUNDLE}")
        expanded = pd.DataFrame(df["embedding_20d"].tolist(), columns=[f"embedding_reduced_{i:02d}" for i in range(1, 21)])
        df = pd.concat([df.drop(columns=["embedding_20d"]), expanded], axis=1)
        emb_cols = _embedding_columns(df)
        source = {"kind": "optimized_cluster_bundle", "path": str(FALLBACK_CLUSTER_BUNDLE)}
    else:
        raise FileNotFoundError(
            "No 20D video embedding artifact found. Run the upstream Graphiko video embedding export first.\n"
            f"Expected one of:\n- {PRIMARY_20D_CSV}\n- {FALLBACK_CLUSTER_BUNDLE}"
        )

    required = {"video_id", "video_title"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required video metadata columns: {sorted(missing)}")

    if "channel_name" not in df.columns:
        if "channel_id" in df.columns:
            df["channel_name"] = df["channel_id"].astype(str)
        else:
            raise ValueError("Need either channel_name or channel_id to group videos by channel.")

    df = df.copy()
    df[emb_cols] = df[emb_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=emb_cols, how="any")
    df["channel_name"] = df["channel_name"].astype(str)
    df["video_title"] = df["video_title"].fillna("").astype(str)

    if df.empty:
        raise ValueError("The 20D embedding table is empty after numeric validation.")

    source["n_videos"] = int(len(df))
    source["n_channels"] = int(df["channel_name"].nunique())
    return df, emb_cols, source

videos_df, EMB_COLS, input_source = load_20d_video_embeddings()
print(json.dumps(input_source, indent=2))
display(videos_df[["channel_name", "video_id", "video_title", *EMB_COLS[:3]]].head())


Create reusable point-cloud objects. Each channel becomes a matrix of shape `n_videos × 20`; downstream sections compute centroid, distribution, overlap, and radial statistics from these point clouds.

In [ ]:
channel_clouds: dict[str, np.ndarray] = {
    channel: group[EMB_COLS].to_numpy(dtype=float)
    for channel, group in videos_df.groupby("channel_name", sort=True)
}
channels = sorted(channel_clouds)

channel_titles: dict[str, list[str]] = {
    channel: group.sort_values("video_id")["video_title"].dropna().astype(str).tolist()
    for channel, group in videos_df.groupby("channel_name", sort=True)
}

print(f"Loaded {len(videos_df):,} videos across {len(channels):,} channels.")
print(pd.Series({c: len(titles) for c, titles in channel_titles.items()}, name="n_videos").describe())

## 2. Generate and cache channel descriptions with native Colab Gemini

For each channel, Gemini receives a compact summary of video titles plus point-cloud statistics and returns structured JSON. Per-channel JSON files are written to Drive and skipped on later runs. An overall cross-channel synthesis is stored separately and also skipped if present.

The cache key is the normalized channel name. This keeps the expensive LLM step reusable for later metric experiments.

In [ ]:
def slugify(value: str, max_len: int = 120) -> str:
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", value.strip()).strip("_")
    if not slug:
        slug = hashlib.sha1(value.encode("utf-8")).hexdigest()[:12]
    return slug[:max_len]


def safe_json_loads(text: str) -> dict[str, Any]:
    """Parse JSON from a model response, accepting fenced blocks and small surrounding prose."""
    cleaned = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", cleaned, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        cleaned = fenced.group(1).strip()
    if not cleaned.startswith("{"):
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start >= 0 and end > start:
            cleaned = cleaned[start:end + 1]
    return json.loads(cleaned)


def cloud_summary_for_prompt(points: np.ndarray) -> dict[str, Any]:
    center = points.mean(axis=0)
    radii = np.linalg.norm(points - center, axis=1)
    std = points.std(axis=0)
    top_dims = np.argsort(-std)[:5]
    return {
        "n_videos": int(points.shape[0]),
        "mean_radius": float(np.mean(radii)),
        "median_radius": float(np.median(radii)),
        "p90_radius": float(np.percentile(radii, 90)),
        "top_variable_20d_dimensions": [int(i + 1) for i in top_dims],
    }


def prompt_for_channel(channel: str, titles: list[str], points: np.ndarray, max_titles: int = 120) -> str:
    sampled_titles = titles[:max_titles]
    title_block = "\n".join(f"- {title[:220]}" for title in sampled_titles)
    stats = json.dumps(cloud_summary_for_prompt(points), indent=2)
    return f"""
You are describing a YouTube channel for a geometric research study.

Channel: {channel}

Video title sample ({len(sampled_titles)} of {len(titles)} videos):
{title_block}

20D point-cloud summary:
{stats}

Return ONLY valid JSON with these keys:
- channel_name: string
- short_description: 2-3 sentences
- detailed_description: 5-8 sentences describing the channel's recurring themes, audience promise, and breadth
- central_topics: array of 5-10 short strings, ordered most central first
- peripheral_topics: array of 3-8 short strings
- style_and_format: array of short strings
- breadth_assessment: one of ["narrow", "moderate", "broad"]
- geometric_expectation: object with keys volume, overlap_likelihood, concentration, all short strings
""".strip()


def generate_text(prompt: str, model_name: str = "google/gemini-2.5-flash") -> str:
    """Use Colab's native Gemini integration. No API key is needed in Colab."""
    response = ai.generate_text(prompt, model_name=model_name)
    return str(response)


def channel_description_path(channel: str) -> Path:
    return CHANNEL_DESCRIPTION_DIR / f"{slugify(channel)}.json"


def load_or_generate_channel_description(channel: str, model_name: str = "google/gemini-2.5-flash") -> dict[str, Any]:
    path = channel_description_path(channel)
    if path.exists():
        with path.open("r", encoding="utf-8") as fh:
            return json.load(fh)

    prompt = prompt_for_channel(channel, channel_titles[channel], channel_clouds[channel])
    raw = generate_text(prompt, model_name=model_name)
    parsed = safe_json_loads(raw)
    parsed.setdefault("channel_name", channel)
    parsed["_metadata"] = {
        "model": model_name,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "source_video_count": len(channel_titles[channel]),
        "input_source": input_source,
    }
    with path.open("w", encoding="utf-8") as fh:
        json.dump(parsed, fh, ensure_ascii=False, indent=2)
    time.sleep(0.5)
    return parsed

channel_descriptions = {}
for channel in tqdm(channels, desc="Channel descriptions"):
    channel_descriptions[channel] = load_or_generate_channel_description(channel)

print(f"Cached descriptions in {CHANNEL_DESCRIPTION_DIR}")
display(pd.DataFrame([
    {
        "channel_name": c,
        "short_description": d.get("short_description", ""),
        "breadth_assessment": d.get("breadth_assessment", ""),
    }
    for c, d in channel_descriptions.items()
]).head())


Generate an overall synthesis after all per-channel descriptions exist. This artifact helps document the global corpus and gives later notebooks a reusable cross-channel summary without re-calling Gemini.

In [ ]:
def description_text(desc: dict[str, Any]) -> str:
    parts = [
        desc.get("short_description", ""),
        desc.get("detailed_description", ""),
        "Central topics: " + ", ".join(desc.get("central_topics", [])),
        "Peripheral topics: " + ", ".join(desc.get("peripheral_topics", [])),
        "Style: " + ", ".join(desc.get("style_and_format", [])),
    ]
    return "\n".join(p for p in parts if p and p.strip())


def load_or_generate_overall_description(model_name: str = "google/gemini-2.5-flash") -> dict[str, Any]:
    if OVERALL_DESCRIPTION_PATH.exists():
        with OVERALL_DESCRIPTION_PATH.open("r", encoding="utf-8") as fh:
            return json.load(fh)

    compact = []
    for channel, desc in channel_descriptions.items():
        compact.append({
            "channel_name": channel,
            "short_description": desc.get("short_description", ""),
            "central_topics": desc.get("central_topics", [])[:8],
            "breadth_assessment": desc.get("breadth_assessment", ""),
        })
    prompt = f"""
You are summarizing a corpus of YouTube channels for a semantic geometry study.

Channel descriptions:
{json.dumps(compact, ensure_ascii=False, indent=2)[:45000]}

Return ONLY valid JSON with:
- corpus_summary: 4-7 sentences
- major_families: array of objects with name, channels, defining_features
- expected_geometry: object describing likely clusters, overlaps, outliers, and broad/narrow regions
- caveats: array of methodological caveats
""".strip()
    parsed = safe_json_loads(generate_text(prompt, model_name=model_name))
    parsed["_metadata"] = {
        "model": model_name,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "n_channels": len(channels),
        "input_source": input_source,
    }
    with OVERALL_DESCRIPTION_PATH.open("w", encoding="utf-8") as fh:
        json.dump(parsed, fh, ensure_ascii=False, indent=2)
    return parsed

overall_description = load_or_generate_overall_description()
print(json.dumps(overall_description, indent=2, ensure_ascii=False)[:2000])


## 3. Compare relative distances: descriptions vs. point-cloud videos

This section builds several distance matrices from the 20D video point clouds and from the generated descriptions. The evaluation asks whether each representation induces similar channel neighborhoods.

Point-cloud metrics:

- **Centroid Euclidean:** distance between each channel's mean 20D vector.
- **Symmetric Chamfer:** average nearest-neighbor distance from cloud A to B and from B to A.
- **Gaussian W2 approximation:** compares each cloud as a Gaussian using mean and covariance.
- **Sliced Wasserstein:** projects clouds onto random directions and averages 1D quantile distances.

Description metrics:

- **Sentence-transformer cosine:** semantic embedding of the whole generated description.
- **TF-IDF cosine:** lexical baseline over the same generated description.

In [ ]:
def matrix_to_frame(matrix: np.ndarray, labels: list[str]) -> pd.DataFrame:
    return pd.DataFrame(matrix, index=labels, columns=labels)


def centroid_distance(clouds: dict[str, np.ndarray], labels: list[str]) -> np.ndarray:
    centroids = np.vstack([clouds[label].mean(axis=0) for label in labels])
    return pairwise_distances(centroids, metric="euclidean")


def symmetric_chamfer_distance(a: np.ndarray, b: np.ndarray) -> float:
    ab = cdist(a, b)
    return float((ab.min(axis=1).mean() + ab.min(axis=0).mean()) / 2)


def pairwise_cloud_metric(clouds: dict[str, np.ndarray], labels: list[str], fn) -> np.ndarray:
    n = len(labels)
    out = np.zeros((n, n), dtype=float)
    for i in tqdm(range(n), desc=getattr(fn, "__name__", "cloud metric")):
        for j in range(i + 1, n):
            value = fn(clouds[labels[i]], clouds[labels[j]])
            out[i, j] = out[j, i] = value
    return out


def sqrtm_psd(mat: np.ndarray) -> np.ndarray:
    vals, vecs = np.linalg.eigh((mat + mat.T) / 2)
    vals = np.clip(vals, 0, None)
    return (vecs * np.sqrt(vals)) @ vecs.T


def gaussian_w2_distance(a: np.ndarray, b: np.ndarray, ridge: float = 1e-6) -> float:
    ma, mb = a.mean(axis=0), b.mean(axis=0)
    ca = np.cov(a, rowvar=False) + np.eye(a.shape[1]) * ridge
    cb = np.cov(b, rowvar=False) + np.eye(b.shape[1]) * ridge
    ca_sqrt = sqrtm_psd(ca)
    middle = sqrtm_psd(ca_sqrt @ cb @ ca_sqrt)
    covariance_term = np.trace(ca + cb - 2 * middle)
    return float(np.sqrt(max(np.sum((ma - mb) ** 2) + covariance_term, 0)))


def sliced_wasserstein_distance(a: np.ndarray, b: np.ndarray, n_dirs: int = 128, seed: int = 13) -> float:
    rng = np.random.default_rng(seed)
    dirs = rng.normal(size=(n_dirs, a.shape[1]))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
    distances = []
    q = np.linspace(0, 1, min(len(a), len(b), 200))
    for direction in dirs:
        pa = np.quantile(a @ direction, q)
        pb = np.quantile(b @ direction, q)
        distances.append(np.mean(np.abs(pa - pb)))
    return float(np.mean(distances))

point_distance_matrices = {
    "point_centroid_euclidean": centroid_distance(channel_clouds, channels),
    "point_symmetric_chamfer": pairwise_cloud_metric(channel_clouds, channels, symmetric_chamfer_distance),
    "point_gaussian_w2": pairwise_cloud_metric(channel_clouds, channels, gaussian_w2_distance),
    "point_sliced_wasserstein": pairwise_cloud_metric(channel_clouds, channels, sliced_wasserstein_distance),
}

for name, matrix in point_distance_matrices.items():
    matrix_to_frame(matrix, channels).to_csv(TABLES_DIR / f"{name}.csv")

print("Saved point-cloud distance matrices.")

Embed each cached description and compute semantic distance matrices. Sentence-transformers provide a dense semantic baseline, while TF-IDF provides an interpretable lexical baseline that is useful for diagnosing whether alignment is driven by repeated words rather than semantics.

In [ ]:
texts = [description_text(channel_descriptions[channel]) for channel in channels]

text_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
description_embeddings = text_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

tfidf = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2), stop_words="english")
tfidf_matrix = tfidf.fit_transform(texts)

description_distance_matrices = {
    "description_sbert_cosine": cosine_distances(description_embeddings),
    "description_tfidf_cosine": cosine_distances(tfidf_matrix),
}

for name, matrix in description_distance_matrices.items():
    matrix_to_frame(matrix, channels).to_csv(TABLES_DIR / f"{name}.csv")

print("Saved description distance matrices.")

Evaluate alignment across every point-cloud metric and description metric. The table includes global matrix correlations, row-wise neighborhood rank agreement, and nearest-neighbor hit rates. These metrics are complementary: a high global correlation can coexist with poor top-neighbor agreement, so both are reported.

In [ ]:
def upper_triangle_values(matrix: np.ndarray) -> np.ndarray:
    idx = np.triu_indices_from(matrix, k=1)
    return matrix[idx]


def rowwise_rank_correlation(a: np.ndarray, b: np.ndarray, labels: list[str]) -> dict[str, float]:
    spears, kendalls = [], []
    for i in range(len(labels)):
        mask = np.arange(len(labels)) != i
        ar = pd.Series(a[i, mask]).rank(method="average").to_numpy()
        br = pd.Series(b[i, mask]).rank(method="average").to_numpy()
        spears.append(spearmanr(ar, br).statistic)
        kendalls.append(kendalltau(ar, br).statistic)
    return {
        "mean_row_spearman": float(np.nanmean(spears)),
        "mean_row_kendall": float(np.nanmean(kendalls)),
    }


def nearest_neighbors(matrix: np.ndarray, i: int, k: int) -> set[int]:
    row = matrix[i].copy()
    row[i] = np.inf
    return set(np.argsort(row)[:k])


def hit_at_k(a: np.ndarray, b: np.ndarray, k: int) -> float:
    hits = []
    for i in range(a.shape[0]):
        hits.append(len(nearest_neighbors(a, i, k) & nearest_neighbors(b, i, k)) / k)
    return float(np.mean(hits))


def top1_match_rate(a: np.ndarray, b: np.ndarray) -> float:
    matches = []
    for i in range(a.shape[0]):
        ar = a[i].copy(); ar[i] = np.inf
        br = b[i].copy(); br[i] = np.inf
        matches.append(int(np.argmin(ar) == np.argmin(br)))
    return float(np.mean(matches))

alignment_rows = []
for p_name, p_matrix in point_distance_matrices.items():
    for d_name, d_matrix in description_distance_matrices.items():
        p_vals = upper_triangle_values(p_matrix)
        d_vals = upper_triangle_values(d_matrix)
        row_corr = rowwise_rank_correlation(p_matrix, d_matrix, channels)
        alignment_rows.append({
            "point_metric": p_name,
            "description_metric": d_name,
            "upper_pearson": float(pearsonr(p_vals, d_vals).statistic),
            "upper_spearman": float(spearmanr(p_vals, d_vals).statistic),
            **row_corr,
            "top1_match_rate": top1_match_rate(p_matrix, d_matrix),
            "hit_at_3": hit_at_k(p_matrix, d_matrix, min(3, len(channels) - 1)),
            "hit_at_5": hit_at_k(p_matrix, d_matrix, min(5, len(channels) - 1)),
        })

alignment_df = pd.DataFrame(alignment_rows).sort_values("upper_spearman", ascending=False)
alignment_df.to_csv(TABLES_DIR / "relative_distance_alignment_summary.csv", index=False)
display(alignment_df)

Visualize the alignment table and the best-performing pair as a scatter plot. The heatmap is useful for comparing metric families; the scatter plot reveals whether a few outliers drive the correlation.

In [ ]:
plt.figure(figsize=(12, 5))
pivot = alignment_df.pivot(index="point_metric", columns="description_metric", values="upper_spearman")
sns.heatmap(pivot, annot=True, cmap="vlag", center=0, fmt=".2f")
plt.title("Relative-distance alignment: upper-triangle Spearman correlation")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "relative_distance_alignment_heatmap.png", dpi=180)
plt.show()

best = alignment_df.iloc[0]
p_matrix = point_distance_matrices[best["point_metric"]]
d_matrix = description_distance_matrices[best["description_metric"]]
plt.figure(figsize=(6, 5))
sns.regplot(x=upper_triangle_values(p_matrix), y=upper_triangle_values(d_matrix), scatter_kws={"alpha": 0.65})
plt.xlabel(best["point_metric"])
plt.ylabel(best["description_metric"])
plt.title("Best metric-pair channel distance scatter")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "best_relative_distance_scatter.png", dpi=180)
plt.show()

## 4. Compare point-cloud shape with semantic description segments

This section splits each generated description into meaningful segments and caches those segments. Gemini is used for segmentation when no cache exists; a deterministic sentence-based fallback is available only for malformed responses. The segment cache makes shape experiments reusable.

Shape concepts tested here:

- **Volume / breadth:** point-cloud covariance volume vs. segment-embedding covariance volume and segment dispersion.
- **Overlap:** point-cloud cross-channel overlap vs. description-segment semantic overlap.
- **Concentration:** average radial concentration in the cloud vs. semantic concentration in segment embeddings.

In [ ]:
def fallback_segments(text: str) -> list[dict[str, Any]]:
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]
    return [
        {"segment_index": i, "segment_text": sentence, "role": "sentence", "centrality_hint": "unknown"}
        for i, sentence in enumerate(sentences)
    ]


def segment_path(channel: str) -> Path:
    return SEGMENT_DIR / f"{slugify(channel)}.json"


def load_or_generate_segments(channel: str, model_name: str = "google/gemini-2.5-flash") -> list[dict[str, Any]]:
    path = segment_path(channel)
    if path.exists():
        with path.open("r", encoding="utf-8") as fh:
            return json.load(fh)["segments"]

    text = description_text(channel_descriptions[channel])
    prompt = f"""
Split the following channel description into semantically meaningful segments for geometry comparison.

Channel: {channel}

Description:
{text}

Return ONLY valid JSON with key `segments`, an array of 5-14 objects. Each object must have:
- segment_index: integer starting at 0 in narrative order
- segment_text: one coherent semantic segment, not a heading
- role: one of ["core_identity", "main_topic", "format_style", "audience", "peripheral_topic", "caveat"]
- centrality_hint: one of ["central", "supporting", "peripheral"]
""".strip()

    try:
        parsed = safe_json_loads(generate_text(prompt, model_name=model_name))
        segments = parsed.get("segments", [])
        if not isinstance(segments, list) or not segments:
            raise ValueError("No segments returned")
    except Exception:
        segments = fallback_segments(text)

    normalized = []
    for i, segment in enumerate(segments):
        segment_text = str(segment.get("segment_text", "")).strip()
        if not segment_text:
            continue
        normalized.append({
            "segment_index": int(segment.get("segment_index", i)),
            "segment_text": segment_text,
            "role": str(segment.get("role", "unknown")),
            "centrality_hint": str(segment.get("centrality_hint", "unknown")),
        })
    normalized = sorted(normalized, key=lambda x: x["segment_index"])

    payload = {
        "channel_name": channel,
        "segments": normalized,
        "_metadata": {
            "model": model_name,
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "description_source": str(channel_description_path(channel)),
        },
    }
    with path.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)
    time.sleep(0.5)
    return normalized

channel_segments = {channel: load_or_generate_segments(channel) for channel in tqdm(channels, desc="Description segments")}
segment_rows = []
for channel, segments in channel_segments.items():
    for segment in segments:
        segment_rows.append({"channel_name": channel, **segment})
segments_df = pd.DataFrame(segment_rows)
segments_df.to_csv(TABLES_DIR / "description_segments.csv", index=False)
display(segments_df.head(12))

Embed the semantic segments and compute per-channel shape descriptors in the same spirit as the 20D point-cloud descriptors. Because each channel has fewer description segments than videos, the log-volume uses a small ridge and PCA fallback to remain numerically stable.

In [ ]:
segment_texts = segments_df["segment_text"].tolist()
segment_embeddings = text_model.encode(segment_texts, normalize_embeddings=True, show_progress_bar=True)
segments_df["_embedding_row"] = np.arange(len(segments_df))

segment_embedding_by_channel = {
    channel: segment_embeddings[segments_df.index[segments_df["channel_name"] == channel].to_numpy()]
    for channel in channels
}


def log_cov_volume(points: np.ndarray, ridge: float = 1e-6) -> float:
    if len(points) < 2:
        return float("nan")
    centered = points - points.mean(axis=0, keepdims=True)
    n_components = min(centered.shape[0] - 1, centered.shape[1])
    if n_components <= 0:
        return float("nan")
    reduced = PCA(n_components=n_components, random_state=13).fit_transform(centered)
    cov = np.cov(reduced, rowvar=False)
    cov = np.atleast_2d(cov) + np.eye(n_components) * ridge
    sign, logdet = np.linalg.slogdet(cov)
    return float(logdet if sign > 0 else np.nan)


def mean_radius(points: np.ndarray) -> float:
    center = points.mean(axis=0, keepdims=True)
    return float(np.linalg.norm(points - center, axis=1).mean())


def concentration_score(points: np.ndarray) -> float:
    radii = np.linalg.norm(points - points.mean(axis=0, keepdims=True), axis=1)
    return float(1 / (np.mean(radii) + 1e-9))

shape_rows = []
for channel in channels:
    cloud = channel_clouds[channel]
    seg = segment_embedding_by_channel[channel]
    shape_rows.append({
        "channel_name": channel,
        "n_videos": len(cloud),
        "n_segments": len(seg),
        "point_log_cov_volume": log_cov_volume(cloud),
        "description_log_cov_volume": log_cov_volume(seg),
        "point_mean_radius": mean_radius(cloud),
        "description_mean_radius": mean_radius(seg),
        "point_concentration": concentration_score(cloud),
        "description_concentration": concentration_score(seg),
    })

shape_df = pd.DataFrame(shape_rows)
shape_df.to_csv(TABLES_DIR / "shape_alignment_by_channel.csv", index=False)
display(shape_df.sort_values("point_log_cov_volume", ascending=False).head())

Compute pairwise overlap proxies. For point clouds, overlap is the fraction of videos that have a cross-channel nearest neighbor closer than that channel's internal median nearest-neighbor distance. For description segments, overlap is the analogous fraction of segments whose cross-channel semantic neighbor is closer than the source channel's internal semantic median.

In [ ]:
def internal_nn_threshold(points: np.ndarray) -> float:
    if len(points) < 2:
        return 0.0
    d = pairwise_distances(points)
    np.fill_diagonal(d, np.inf)
    return float(np.median(d.min(axis=1)))


def overlap_fraction(a: np.ndarray, b: np.ndarray, threshold_a: float, threshold_b: float) -> float:
    d = cdist(a, b, metric="euclidean")
    a_overlap = (d.min(axis=1) <= threshold_a).mean() if len(a) else 0
    b_overlap = (d.min(axis=0) <= threshold_b).mean() if len(b) else 0
    return float((a_overlap + b_overlap) / 2)

point_thresholds = {channel: internal_nn_threshold(channel_clouds[channel]) for channel in channels}
segment_thresholds = {channel: internal_nn_threshold(segment_embedding_by_channel[channel]) for channel in channels}

point_overlap = np.zeros((len(channels), len(channels)))
description_overlap = np.zeros_like(point_overlap)
for i, a in enumerate(channels):
    for j, b in enumerate(channels):
        if i == j:
            point_overlap[i, j] = description_overlap[i, j] = 1.0
        elif i < j:
            point_value = overlap_fraction(channel_clouds[a], channel_clouds[b], point_thresholds[a], point_thresholds[b])
            desc_value = overlap_fraction(segment_embedding_by_channel[a], segment_embedding_by_channel[b], segment_thresholds[a], segment_thresholds[b])
            point_overlap[i, j] = point_overlap[j, i] = point_value
            description_overlap[i, j] = description_overlap[j, i] = desc_value

matrix_to_frame(point_overlap, channels).to_csv(TABLES_DIR / "point_cloud_overlap.csv")
matrix_to_frame(description_overlap, channels).to_csv(TABLES_DIR / "description_segment_overlap.csv")

shape_summary = pd.DataFrame([
    {
        "comparison": "volume_log_cov",
        "pearson": pearsonr(shape_df["point_log_cov_volume"], shape_df["description_log_cov_volume"]).statistic,
        "spearman": spearmanr(shape_df["point_log_cov_volume"], shape_df["description_log_cov_volume"]).statistic,
    },
    {
        "comparison": "mean_radius_dispersion",
        "pearson": pearsonr(shape_df["point_mean_radius"], shape_df["description_mean_radius"]).statistic,
        "spearman": spearmanr(shape_df["point_mean_radius"], shape_df["description_mean_radius"]).statistic,
    },
    {
        "comparison": "concentration",
        "pearson": pearsonr(shape_df["point_concentration"], shape_df["description_concentration"]).statistic,
        "spearman": spearmanr(shape_df["point_concentration"], shape_df["description_concentration"]).statistic,
    },
    {
        "comparison": "cross_channel_overlap",
        "pearson": pearsonr(upper_triangle_values(point_overlap), upper_triangle_values(description_overlap)).statistic,
        "spearman": spearmanr(upper_triangle_values(point_overlap), upper_triangle_values(description_overlap)).statistic,
    },
])
shape_summary.to_csv(TABLES_DIR / "shape_alignment_summary.csv", index=False)
display(shape_summary)

Plot the shape-alignment diagnostics. These plots make it easy to see whether volume, concentration, or overlap relationships are monotonic or dominated by a few channels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.regplot(data=shape_df, x="point_log_cov_volume", y="description_log_cov_volume", ax=axes[0])
axes[0].set_title("Volume alignment")
sns.regplot(data=shape_df, x="point_mean_radius", y="description_mean_radius", ax=axes[1])
axes[1].set_title("Breadth / radius alignment")
sns.regplot(data=shape_df, x="point_concentration", y="description_concentration", ax=axes[2])
axes[2].set_title("Concentration alignment")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "shape_alignment_scatterplots.png", dpi=180)
plt.show()

plt.figure(figsize=(6, 5))
sns.regplot(x=upper_triangle_values(point_overlap), y=upper_triangle_values(description_overlap), scatter_kws={"alpha": 0.65})
plt.xlabel("Point-cloud overlap proxy")
plt.ylabel("Description-segment overlap proxy")
plt.title("Cross-channel overlap alignment")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "overlap_alignment_scatter.png", dpi=180)
plt.show()

## 5. Centrality and ordering in segmented descriptions

The final analysis tests whether central ideas appear earlier and more prominently. For each channel, segment centrality is measured as inverse distance to that channel's semantic segment centroid. We correlate segment order with centrality and compare the first segment's centrality to the channel average.

Interpretation guide:

- A **negative Spearman correlation** between `segment_index` and centrality means earlier segments are more central.
- A **positive first-segment lift** means the opening concept is more central than the average segment.

In [ ]:
centrality_rows = []
for channel in channels:
    channel_mask = segments_df["channel_name"] == channel
    channel_segment_rows = segments_df[channel_mask].sort_values("segment_index")
    emb = segment_embedding_by_channel[channel]
    center = emb.mean(axis=0, keepdims=True)
    distances = np.linalg.norm(emb - center, axis=1)
    centrality = -distances  # higher is more central
    order = channel_segment_rows["segment_index"].to_numpy(dtype=float)
    if len(order) >= 3:
        order_spearman = spearmanr(order, centrality).statistic
        order_kendall = kendalltau(order, centrality).statistic
    else:
        order_spearman = np.nan
        order_kendall = np.nan
    centrality_rows.append({
        "channel_name": channel,
        "n_segments": len(order),
        "order_vs_centrality_spearman": float(order_spearman),
        "order_vs_centrality_kendall": float(order_kendall),
        "first_segment_centrality": float(centrality[0]) if len(centrality) else np.nan,
        "mean_segment_centrality": float(np.mean(centrality)) if len(centrality) else np.nan,
        "first_segment_centrality_lift": float(centrality[0] - np.mean(centrality)) if len(centrality) else np.nan,
        "most_central_segment_index": int(order[np.argmax(centrality)]) if len(order) else -1,
    })

centrality_df = pd.DataFrame(centrality_rows)
centrality_df.to_csv(TABLES_DIR / "centrality_ordering_by_channel.csv", index=False)

centrality_summary = pd.DataFrame([{
    "mean_order_vs_centrality_spearman": centrality_df["order_vs_centrality_spearman"].mean(),
    "median_order_vs_centrality_spearman": centrality_df["order_vs_centrality_spearman"].median(),
    "share_negative_order_correlation": (centrality_df["order_vs_centrality_spearman"] < 0).mean(),
    "mean_first_segment_centrality_lift": centrality_df["first_segment_centrality_lift"].mean(),
    "share_positive_first_segment_lift": (centrality_df["first_segment_centrality_lift"] > 0).mean(),
}])
centrality_summary.to_csv(TABLES_DIR / "centrality_ordering_summary.csv", index=False)

display(centrality_summary)
display(centrality_df.sort_values("order_vs_centrality_spearman").head(10))

Plot the centrality-ordering distribution. The vertical line at zero separates channels whose descriptions open centrally (left/negative order correlation) from channels whose later segments are more central (right/positive order correlation).

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.histplot(centrality_df["order_vs_centrality_spearman"].dropna(), bins=15, kde=True)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Spearman(segment order, semantic centrality)")
plt.title("Do channel descriptions move from central ideas outward?")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "centrality_ordering_distribution.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 4.5))
sns.scatterplot(data=centrality_df, x="n_segments", y="first_segment_centrality_lift")
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Opening segment centrality lift")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "first_segment_centrality_lift.png", dpi=180)
plt.show()

## 6. Write manifest and final report tables

The manifest records every input, cache directory, table, plot, model, and timestamp. It is the first file to inspect when reusing or auditing a run.

In [ ]:
manifest = {
    "schema_name": "graphiko.research.semantic_description_pointcloud_alignment",
    "schema_version": "1.0.0",
    "run_started_at": RUN_STARTED_AT,
    "run_completed_at": datetime.now(timezone.utc).isoformat(),
    "input_source": input_source,
    "n_channels": len(channels),
    "n_videos": int(len(videos_df)),
    "native_gemini_integration": "google.colab.ai.generate_text",
    "description_cache_dir": str(CHANNEL_DESCRIPTION_DIR),
    "overall_description_path": str(OVERALL_DESCRIPTION_PATH),
    "segment_cache_dir": str(SEGMENT_DIR),
    "tables_dir": str(TABLES_DIR),
    "plots_dir": str(PLOTS_DIR),
    "tables": sorted(p.name for p in TABLES_DIR.glob("*.csv")),
    "plots": sorted(p.name for p in PLOTS_DIR.glob("*.png")),
    "interpretation_notes": [
        "Distance alignment is strongest when upper-triangle and row-wise rank metrics agree.",
        "Shape alignment is exploratory because descriptions have far fewer points than video clouds.",
        "Negative order-vs-centrality correlation supports central ideas appearing earlier.",
    ],
}
manifest_path = RESULTS_DIR / "manifest.json"
with manifest_path.open("w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)

print(f"Wrote manifest: {manifest_path}")
print(json.dumps(manifest, indent=2)[:3000])